# 07 · From raw exports to the board sentence — Briefs A, B and D as one story

**Produces:** `data/clean/story_visits.csv.gz` (one row per visit, the single table every number below comes from), `data/clean/story_decisions.csv`, `data/clean/story_claims.csv`, and three charts in `slides/`.

**Needs:** the data pack in `data/raw/`. Nothing else: this notebook starts from the raw files and does its own cleaning, so it runs cold.

## The story in one picture

```
   RAW EXPORTS (6 systems, 2 booking systems, 3 id families)
        │   Part 1  CLEAN: one visit = one row; every rule logged with its effect
        ▼
   A · THE COMPANY    "Tests +5% against a plan of +18%. Where did the gap go?"
        │              tree: people x visits per person x tests per visit; then city; then channel
        │              → the gap sits in the CLINIC channel, and Chennai + Jaipur are over-represented
        ▼
   B · TWO CITIES     "Chennai and Jaipur fell while their booking system changed. Real or counting?"
        │              the same visits seen through four lenses, two of which the migration cannot touch
        │              → how much is counting, how much is real, and when it started
        ▼
   D · ONE CLINIC     "CHE-03 (in Chennai) has the worst no-show rate. Real, noise, or not yet knowable?"
                       definition, denominators, chance reference, like-with-like, and what it costs
                       → and how big that is next to the gap in A
```

Every section follows MAP, DO, SEE, CHECK, SUM. Numbers in the SUM cells are printed by code, so the text can never drift from the data.

## CONFIG

In [ ]:
import sys, pathlib, time, warnings
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from kalpa import loaders, checks
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

CONFIG = {
    "raw": ROOT / "data" / "raw", "clean": ROOT / "data" / "clean", "slides": ROOT / "slides",
    # half-open windows [start, end): a booking at 14:00 on 30 June belongs to the half. 'between(a, "06-30")' would drop it.
    "halves": {"H1_2025": ("2025-01-01", "2025-07-01"), "H1_2026": ("2026-01-01", "2026-07-01")},
    "migrated": ["Chennai", "Jaipur"],            # cities that moved to the new booking system on 1 April 2026
    "seed": 20261019,
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True); CONFIG["slides"].mkdir(exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
INK, PINK, VIOLET, GREY, GOLD = "#1A1440", "#D63A6A", "#4E38AB", "#9A97B0", "#C9A227"
check = checks.check
T_START = time.time()

pct = lambda new, old: (new / old - 1) * 100
def half_of(ts: pd.Series) -> pd.Series:
    out = pd.Series("other", index=ts.index)
    for name, (a, b) in CONFIG["halves"].items():
        out[(ts >= a) & (ts < b)] = name
    return out
print("pandas", pd.__version__, "| numpy", np.__version__)

# PART 1 · Clean: turn six exports into one trustworthy table of visits

Before any growth rate, ask what we are counting (handbook, "first principles"): *a test needs a booking, a collected sample and a result — which table records which?* The booking systems record **intent** (a booking with its tests). The collection app records **what happened** at the door (collected or no-show). The lab records **what was run**. We count visits from bookings, and use the other two as independent witnesses.

**What this part does**
1. Load both booking systems at booking-test grain and put them on one footing (city spelling, repeated headers).
2. Repair or flag the records the starter kit would have thrown away, and prove why.
3. Collapse a rescheduled booking into one visit.
4. Attach what happened (collections) and what was run (lab results); give every patient one person id.
5. Reconcile: raw rows → visits, with the headline recomputed after every step.

## 1.1 · MAP and DO: load both booking systems at the same grain

In [ ]:
legacy_files = sorted((CONFIG["raw"] / "bookings" / "legacy").rglob("*.csv.gz"))
nova_files = sorted((CONFIG["raw"] / "bookings" / "nova").rglob("*.jsonl.gz"))
t0 = time.time()
leg = loaders.read_many(legacy_files, "legacy")     # Latin-1, day-first dates, repeated headers dropped by the loader
nov = loaders.read_many(nova_files, "nova")         # JSON lines, tests nested, UTC converted to India time by the loader
HEADER_LINES = leg.attrs["repeated_header_lines"]
print(f"legacy: {len(legacy_files)} files, {len(leg):,} booking-test rows | nova: {len(nova_files)} files, {len(nov):,} rows | {time.time()-t0:.0f}s")
print("repeated header lines dropped from legacy files:", HEADER_LINES)

rows = pd.concat([leg, nov], ignore_index=True)
for c in ["booking_id", "city", "clinic_id", "channel", "source", "patient_id", "test_code", "rescheduled_from", "status_at_export", "system"]:
    rows[c] = rows[c].astype("string")
rows["booking_id"] = rows["booking_id"].str.strip()
N_BANGALORE = int((rows.city == "Bangalore").sum())
rows["city"] = rows["city"].replace({"Bangalore": "Bengaluru"})
N_RAW_ROWS = len(rows)
print("rows with the old spelling 'Bangalore':", f"{N_BANGALORE:,}", "| cities now:", sorted(rows.city.unique()))

### SEE · the grain trap, and the clock check

In [ ]:
sys_view = rows.groupby("system").agg(rows=("booking_id", "size"), bookings=("booking_id", "nunique"))
sys_view["rows_per_booking"] = sys_view.rows / sys_view.bookings
display(sys_view)

# If the UTC to India conversion in the nova loader were wrong, nova slots would sit 5.5 hours off the clinic day.
hours = rows.assign(hr=rows.slot_at.dt.hour).groupby("system").hr.agg(["min", "max"])
display(hours)
check("nova and legacy slots share the same operating hours (time zone handled)", bool(hours.loc["nova", "min"] >= hours.loc["legacy", "min"] and hours.loc["nova", "max"] <= hours.loc["legacy", "max"]), hours.to_dict("index"))

### SUM
Rows are **booking-tests**, not bookings: a count of rows is a count of tests, a count of distinct `booking_id` is a count of visits. Mixing the two is the first way to get this wrong. Both systems now share one schema.

## 1.2 · The records the starter kit rejects — and why we keep them

The starter validator rejects 3,107 bookings. Before accepting that, ask the independent witness: **do the collection app and the lab know about them?** If yes, they are real visits and rejecting them removes real tests from the count.

In [ ]:
ID_OK = r"^(?:BK-[A-Z]{3}-\d{6}-\d{6}|nova_[a-z]{3}_\d{7})$"
rows["bad_id"] = ~rows.booking_id.str.match(ID_OK)
rows["blank_code"] = rows.test_code.isna() | (rows.test_code.str.strip() == "")
rows["created_after_slot"] = rows.created_at > rows.slot_at

def repair_id(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    return s.where(s.str.match(ID_OK), s.str[:-1])          # a 21-char id is a valid 20-char id with a stray '0' appended

# Witness 1: the collection app. Loaded once here; reused in 1.4. The same id repair is applied to it.
coll_raw = pd.concat([pd.read_csv(p, dtype="string") for p in sorted((CONFIG["raw"] / "collections").glob("*.csv.gz"))], ignore_index=True)
coll_raw["booking_id"] = repair_id(coll_raw["booking_id"])
coll_ids = set(coll_raw.booking_id)

bk_flag = rows.assign(bid=repair_id(rows.booking_id)).groupby("bid", sort=False)[["bad_id", "blank_code", "created_after_slot"]].max()
witness = pd.DataFrame({"flag": ["bad id (21 chars, one extra digit)", "blank test code", "created_at after slot_at"],
                        "bookings": [int(bk_flag[c].sum()) for c in bk_flag.columns],
                        "known to the collection app": [int(bk_flag.index[bk_flag[c]].isin(coll_ids).sum()) for c in bk_flag.columns]})
witness["share known %"] = witness["known to the collection app"] / witness["bookings"] * 100
display(witness)
N_FLAGGED = int(bk_flag.any(axis=1).sum())
print("bookings carrying at least one flag:", f"{N_FLAGGED:,}")

### The repair rule, stated and checked

In [ ]:
print("bad id example:", rows.loc[rows.bad_id, "booking_id"].iloc[0], "| lengths:", rows.loc[rows.bad_id, "booking_id"].str.len().unique().tolist())
orig_valid = set(rows.loc[~rows.bad_id, "booking_id"])
fixed = rows.loc[rows.bad_id, "booking_id"].str[:-1]
check("a repaired id never collides with an id that was valid", len(set(fixed) & orig_valid) == 0, f"{fixed.nunique()} repaired ids")
check("the stray character is always '0'", bool((rows.loc[rows.bad_id, "booking_id"].str[-1] == "0").all()))
N_ID_ROWS = int(rows.bad_id.sum())
rows.loc[rows.bad_id, "booking_id"] = fixed
check("every booking id now matches its pattern", bool(rows.booking_id.str.match(ID_OK).all()))
dup_mask = rows.duplicated(["booking_id", "test_code", "slot_at"]) & ~rows.blank_code
N_EXACT_DUP = int(dup_mask.sum())
rows = rows[~dup_mask].copy()
print("exact duplicate booking-test rows dropped:", N_EXACT_DUP)

### And which of `created_at` / `slot_at` is wrong when they disagree?
`created_at` after `slot_at` is impossible. The collection app timestamps the sample at the door, so it can referee.

In [ ]:
x = rows[rows.created_after_slot].drop_duplicates("booking_id")[["booking_id", "slot_at", "created_at"]]
cc = coll_raw.drop_duplicates("booking_id")[["booking_id", "collected_at"]].assign(collected_at=lambda d: pd.to_datetime(d.collected_at, errors="coerce"))
x = x.merge(cc, on="booking_id")
hrs_slot = (x.collected_at - x.slot_at).dt.total_seconds() / 3600
hrs_created = (x.created_at - x.slot_at).dt.total_seconds() / 3600
print(f"{len(x):,} bookings. Sample collected relative to slot_at: median {hrs_slot.median():+.1f} h.   created_at minus slot_at: median {hrs_created.median():+.1f} h")
check("slot_at is the trustworthy timestamp (collection happens at the slot)", abs(hrs_slot.median()) < 3, f"median {hrs_slot.median():+.2f} h")

### SUM
All three flags describe **real visits**: the collection app saw almost every one of them. The kit rejects them anyway. We therefore **repair the id** (drop the stray 0), **keep blank-test-code rows** (the test happened; its code and price are unknown) and **keep the visit whose `created_at` is wrong** (`slot_at` is corroborated by the sample timestamp; `created_at` is not used anywhere below). The effect on the headline is measured in 1.6.

## 1.3 · One visit = one booking record
Roll booking-test rows up to **booking grain**: one row per booking with `n_tests`. Then settle what a rescheduled nova booking is.

In [ ]:
g = rows.groupby("booking_id", sort=False)
bk = g.agg(system=("system", "first"), slot_at=("slot_at", "first"), created_at=("created_at", "first"), city=("city", "first"),
           clinic_id=("clinic_id", "first"), channel=("channel", "first"), source=("source", "first"), patient_id=("patient_id", "first"),
           home_fee=("home_fee", "first"), rescheduled_from=("rescheduled_from", "first"), status_at_export=("status_at_export", "first"),
           n_tests=("test_code", "size"), list_value=("test_price", "sum"),
           bad_id=("bad_id", "max"), blank_code=("blank_code", "max"), created_after_slot=("created_after_slot", "max")).reset_index()
check("booking table has one row per booking id", bk.booking_id.is_unique, f"{len(bk):,} bookings from {len(rows):,} booking-test rows")
check("slot_at is constant inside a booking", int(g.slot_at.nunique().max()) == 1)
print(bk.groupby("system").size().to_dict())

### Did the grain conversion lose any booking?
Nova nests tests inside each booking. A booking whose test list is empty becomes **zero** booking-test rows, so it would vanish without a trace. Count both sides first.

In [ ]:
import gzip, json
nova_raw, empties = 0, []
for p in nova_files:
    with gzip.open(p, "rt", encoding="utf-8") as f:
        for ln in f:
            r = json.loads(ln); nova_raw += 1
            if not r["tests"]: empties.append(r)
n_nova_loaded = int((bk.system == "nova").sum())
print(f"nova bookings in the files: {nova_raw:,} | in the booking table: {n_nova_loaded:,} | with an empty tests list: {len(empties)}")
emp = pd.DataFrame([{"booking_id": r["booking_id"], "system": "nova", "slot_at": r["slot_at"], "created_at": r["created_at"], "city": r["city"], "clinic_id": r["clinic_id"],
                     "channel": r["channel"].lower(), "source": r["source"].lower(), "patient_id": r["patient"]["patient_id"], "home_fee": r["home_fee"],
                     "rescheduled_from": r.get("rescheduled_from"), "status_at_export": r.get("status_at_export"), "n_tests": 0, "list_value": 0,
                     "bad_id": False, "blank_code": False, "created_after_slot": False} for r in empties])
for c in ["slot_at", "created_at"]:
    emp[c] = pd.to_datetime(emp[c], utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
for c in ["booking_id", "system", "city", "clinic_id", "channel", "source", "patient_id", "rescheduled_from", "status_at_export"]:
    emp[c] = emp[c].astype("string")
N_EMPTY_TESTS = len(emp)
bk = pd.concat([bk, emp], ignore_index=True)
check("every nova booking in the files is now in the booking table", int((bk.system == "nova").sum()) == nova_raw, f"{nova_raw:,}")
check("booking table still has one row per booking id", bk.booking_id.is_unique)

### Trace one rescheduled visit by hand (handbook, brief B step 2)

In [ ]:
succ = bk[bk.rescheduled_from.notna()]
orig = bk[bk.booking_id.isin(succ.rescheduled_from)]          # linked by id, not by the status flag
print("status_at_export on those originals:", orig.status_at_export.value_counts(dropna=False).to_dict())
pair = succ.merge(orig, left_on="rescheduled_from", right_on="booking_id", suffixes=("_new", "_old"))
print(f"{len(succ):,} successor records point at {succ.rescheduled_from.nunique():,} originals; {len(pair):,} pairs resolve")
one = pair.iloc[0]
print(pd.DataFrame({"original": [one.booking_id_old, one.slot_at_old, one.patient_id_old, one.clinic_id_old, one.n_tests_old],
                    "successor": [one.booking_id_new, one.slot_at_new, one.patient_id_new, one.clinic_id_new, one.n_tests_new]},
                   index=["booking_id", "slot_at", "patient", "clinic", "n_tests"]))
check("every successor resolves to exactly one original", len(pair) == len(succ) == succ.rescheduled_from.nunique())
known = pair.n_tests_old > 0
check("same patient and clinic in every pair", bool((pair.patient_id_new == pair.patient_id_old).all() and (pair.clinic_id_new == pair.clinic_id_old).all()))
check("same number of tests in every pair whose original lists its tests", bool((pair.n_tests_new[known] == pair.n_tests_old[known]).all()), f"{int((~known).sum())} original(s) with an empty test list")
check("no chains: a successor is never itself rescheduled again", not succ.booking_id.isin(succ.rescheduled_from).any())

### Which of the pair did the sample get collected against?

In [ ]:
coll_raw["collected_at"] = pd.to_datetime(coll_raw.collected_at, errors="coerce")
cl1 = coll_raw.drop_duplicates(["booking_id", "status", "collected_at"])
for_pair = pair.merge(cl1[["booking_id", "collected_at", "status"]], left_on="booking_id_old", right_on="booking_id", how="inner", suffixes=("", "_c"))
d_old = (for_pair.collected_at - for_pair.slot_at_old).dt.total_seconds() / 86400
d_new = (for_pair.collected_at - for_pair.slot_at_new).dt.total_seconds() / 86400
print(f"{len(for_pair):,} pairs have a collection row. Days from collection to ORIGINAL slot: median {d_old.median():+.2f}.  To SUCCESSOR slot: median {d_new.median():+.2f}")
print("collection rows attached to a successor id:", int(succ.booking_id.isin(coll_ids).sum()))
check("collections sit on the original booking, at the original slot", abs(d_old.median()) < 0.5 and succ.booking_id.isin(coll_ids).sum() == 0)

In [ ]:
# THE RULE: one visit = the root booking. Keep the record the collection app and the lab point to; drop the successor.
visits = bk[bk.rescheduled_from.isna()].copy()
visits["was_rescheduled"] = visits.booking_id.isin(succ.rescheduled_from)
# an original whose test list is empty takes the tests of its successor (same visit, same tests)
inherit = visits.was_rescheduled & (visits.n_tests == 0)
sm = succ.set_index("rescheduled_from")
visits.loc[inherit, "n_tests"] = visits.loc[inherit, "booking_id"].map(sm.n_tests).to_numpy()
visits.loc[inherit, "list_value"] = visits.loc[inherit, "booking_id"].map(sm.list_value).to_numpy()
N_INHERIT_TESTS = int(visits.loc[inherit, "n_tests"].sum())
check("bookings = visits + successor records", len(bk) == len(visits) + len(succ), f"{len(bk):,} = {len(visits):,} + {len(succ):,}")
print("rescheduled visits by city:", visits[visits.was_rescheduled].groupby("city").size().to_dict())

### SUM
A reschedule in the new system writes a **second record** for the same visit; the first record keeps the sample and the lab results. The starter kit drops the first and keeps the second, the one the collection app has never heard of. We do the opposite: keep the original, drop the successor. The visit count is identical either way, but the **status** is not (see 1.5).

## 1.4 · Attach what happened at the door (collections)
The collection app has one row per sample event. It is the only table that says **collected** or **no-show**. Two traps: the vendor scanned some samples twice, and a booking with no collection row is neither of those — it is *unknown* (cancelled, or never reached the app).

In [ ]:
coll = coll_raw.drop_duplicates(["booking_id", "status", "collected_at"]).copy()
N_DOUBLE_SCANS = len(coll_raw) - len(coll)
checks.reconciles("collections: double scans removed", len(coll_raw), len(coll), N_DOUBLE_SCANS)
check("one collection row per booking after de-duplication", coll.booking_id.is_unique)
N_COLL_ORPHANS = int((~coll.booking_id.isin(bk.booking_id)).sum())
print("collection rows whose booking id is in no booking system:", N_COLL_ORPHANS)

before = visits
visits = visits.merge(coll[["booking_id", "status", "collected_at", "phlebotomist_id"]], on="booking_id", how="left", validate="one_to_one")
checks.join_did_not_fan_out("visits LEFT JOIN collections", before, visits, "booking_id")
visits["status"] = visits["status"].astype("object").fillna("no_collection_record")
skew = visits.collected_at < visits.slot_at - pd.Timedelta(days=1)
visits["flag_clock_skew"] = skew
N_SKEW = int(skew.sum())
print("sample timestamped more than a day before the slot (clock skew, flagged and kept):", N_SKEW)
display(visits.status.value_counts().to_frame("visits"))

### Break it on purpose · what the starter kit's reschedule rule does to a status
The kit drops the *original* of a rescheduled pair and keeps the successor. The successor has no collection row, so the visit is labelled "no collection record" — a cancellation that never happened.

In [ ]:
bk_s = bk[~bk.booking_id.isin(succ.rescheduled_from)].merge(coll[["booking_id"]].assign(has_row=True), on="booking_id", how="left")
bk_s["half"] = half_of(bk_s.slot_at)
starter_share = bk_s[bk_s.half == "H1_2026"].groupby("city").has_row.apply(lambda s: s.isna().mean() * 100)
ours_share = visits[half_of(visits.slot_at) == "H1_2026"].groupby("city").status.apply(lambda s: (s == "no_collection_record").mean() * 100)
cmp_status = pd.DataFrame({"starter rule %": starter_share, "our rule %": ours_share})
display(cmp_status)
others = [c for c in cmp_status.index if c not in CONFIG["migrated"]]
check("with our rule Chennai and Jaipur look like the other four cities on 'no collection record'",
      bool((cmp_status.loc[CONFIG["migrated"], "our rule %"] - cmp_status.loc[others, "our rule %"].mean()).abs().max() < 1.0),
      f"ours {cmp_status.loc[CONFIG['migrated'], 'our rule %'].round(1).to_dict()} vs others {cmp_status.loc[others, 'our rule %'].mean():.1f}")

### SUM
Collections are attached with row counts checked on both sides. About 3% of visits have no collection row in every city; the kit's rule would have shown about twice that in Chennai and Jaipur, a made-up jump that lands exactly where Brief B is looking.

## 1.5 · Attach what was run (lab results) — the second independent witness
The lab system never sees the booking system, so a migration cannot change what it records. We join it to visits to get **tests resulted**, and keep it for Part 3.

In [ ]:
res = pd.concat([pd.read_csv(p, dtype="string") for p in sorted((CONFIG["raw"] / "results").rglob("*.csv.gz"))], ignore_index=True)
res["booking_id"] = repair_id(res["booking_id"])
N_RES_RAW = len(res)
res_dup = res.duplicated(["booking_id", "test_code"])
res = res[~res_dup]
checks.reconciles("lab results: duplicate test rows removed", N_RES_RAW, len(res), int(res_dup.sum()))
res_n = res.groupby("booking_id").size()
N_RES_ORPHANS = int((~res.booking_id.isin(visits.booking_id)).sum())
print("result rows whose booking id is not a visit:", N_RES_ORPHANS, "| result rows on a successor id:", int(res.booking_id.isin(succ.booking_id).sum()))
orph = res[~res.booking_id.isin(visits.booking_id)]
orph_serial = orph.booking_id.str.extract(r"-(\d{6})$")[0].astype(float)
N_GHOST_RES = len(orph)
print(f"{N_GHOST_RES:,} lab rows ({N_GHOST_RES/len(res)*100:.2f}% of all) belong to ids found in no booking export; {int((orph_serial >= 900000).sum()):,} of them have a serial of 900000 or more (real serials stop far below that).")
print("by city code:", orph.booking_id.str.extract(r"^(?:BK-)?([A-Za-z]{3})")[0].str.upper().value_counts().to_dict())
orph.to_csv(CONFIG["clean"] / "story_lab_rows_without_booking.csv", index=False)
visits["tests_resulted"] = visits.booking_id.map(res_n).fillna(0).astype(int)
check("no results on no-show visits", int(visits.loc[visits.status == "no_show", "tests_resulted"].sum()) == 0)
cov = visits[(visits.status == "collected") & (half_of(visits.slot_at) != "other")].groupby("system").agg(tests_booked=("n_tests", "sum"), tests_resulted=("tests_resulted", "sum"))
cov["resulted per 100 booked"] = cov.tests_resulted / cov.tests_booked * 100
display(cov)

### SUM
About 97 of every 100 tests booked on a collected visit have a lab result, in both systems. The gap (results not yet posted, or lost) is the same size on either side of the migration, so the lab is a fair witness.

## 1.6 · One person, one id
Patients appear under several ids (typos, re-registration). A growth rate on "patients" needs **people**. Same rule as the starter kit: normalised phone + normalised name + date of birth. The stakes here are low (a slightly wrong frequency); for a results system, merging two humans would be a wrong report sent to the wrong person.

In [ ]:
pat = pd.read_csv(CONFIG["raw"] / "patients" / "patients.csv.gz", dtype="string")
pat["phone_norm"] = loaders.normalise_phone(pat["phone"])
pat["name_norm"] = pat["patient_name"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
pat["person_key"] = pat.groupby(["phone_norm", "name_norm", "dob"], dropna=False).ngroup()
first = ~pat.duplicated("person_key")
N_PAT_DUP = int((~first).sum())
checks.reconciles("patients de-duplicated by phone + name + dob", len(pat), int(first.sum()), N_PAT_DUP)
N_PLACEHOLDER_DOB = int(pat["dob"].eq("1900-01-01").sum())
print("placeholder dob (1900-01-01) rows, kept and not used as a key on its own:", N_PLACEHOLDER_DOB)
id_map = pat[["patient_id", "person_key"]].merge(pat.loc[first, ["person_key", "patient_id"]].rename(columns={"patient_id": "person_id"}), on="person_key", validate="many_to_one")[["patient_id", "person_id"]]
before = visits
visits = visits.merge(id_map, on="patient_id", how="left", validate="many_to_one")
checks.join_did_not_fan_out("visits LEFT JOIN patient map", before, visits, "booking_id")
check("every visit maps to a person", bool(visits.person_id.notna().all()), int(visits.person_id.isna().sum()))
pat["signup"] = pd.to_datetime(pat["signup_date"])
person_signup = pat.groupby("person_key").signup.min()
visits["signup"] = visits.patient_id.map(pat.set_index("patient_id").person_key).map(person_signup)

## 1.7 · Reconcile: raw rows to visits, with the headline after every step
The headline is **tests booked** (every booked test, whatever happened at the door) because that is the quantity the plan was written in; Part 2 proves it. The ledger below is the bridge Dr Menon's finance controller will ask for.

In [ ]:
visits["half"] = half_of(visits.slot_at)
visits["quarter"] = visits.slot_at.dt.to_period("Q").astype(str)
visits["month"] = visits.slot_at.dt.to_period("M").astype(str)

LEDGER = []
def snap(step, df, tests=None, closed_window=False, records=None):
    if closed_window:    # how the starter kit cuts a half: between('2025-01-01','2025-06-30') stops at midnight on 30 June
        h = pd.Series("other", index=df.index)
        h[df.slot_at.between("2025-01-01", "2025-06-30")] = "H1_2025"; h[df.slot_at.between("2026-01-01", "2026-06-30")] = "H1_2026"
    else:
        h = half_of(df.slot_at)
    t = df.groupby(h)[tests].sum() if tests else df.groupby(h).size()
    a, b = int(t.get("H1_2025", 0)), int(t.get("H1_2026", 0))
    LEDGER.append({"step": step, "records": records if records is not None else len(df), "tests H1 2025": a, "tests H1 2026": b, "growth %": pct(b, a)})

snap("A  raw booking-test rows (successor records included)", rows)
snap("B  one visit per booking (successor records dropped)  <- OUR TABLE", visits, tests="n_tests")
flag_ids = set(bk.booking_id[bk.bad_id | bk.blank_code | bk.created_after_slot])
starter = bk[~bk.booking_id.isin(succ.rescheduled_from) & ~bk.booking_id.isin(flag_ids)]
snap("C  as B, but rejecting the 3,107 flagged bookings (starter validator)", starter, tests="n_tests")
snap("D  as C, and the half ends at midnight on 30 June (starter date window)", starter, tests="n_tests", closed_window=True)
ledger = pd.DataFrame(LEDGER)
display(ledger)
G = {r["step"][0]: r["growth %"] for r in LEDGER}

In [ ]:
check("tests in the booking table = booking-test rows", int(bk.n_tests.sum()) == len(rows), f"{int(bk.n_tests.sum()):,} = {len(rows):,}")
check("rows = visits' tests + successor records' tests", len(rows) == int(visits.n_tests.sum()) - N_INHERIT_TESTS + int(bk.loc[bk.booking_id.isin(succ.booking_id), "n_tests"].sum()))
check("visits table has one row per visit", visits.booking_id.is_unique, f"{len(visits):,} visits")
checks.reconciles("bookings to visits", len(bk), len(visits), len(succ))

In [ ]:
DECISIONS = pd.DataFrame([
    ("legacy files", f"{HEADER_LINES} repeated header lines inside monthly files", "drop", "append artefact, not data", HEADER_LINES, "none"),
    ("city", f"'Bangalore' and 'Bengaluru' both used ({N_BANGALORE:,} rows)", "map to Bengaluru", "one city, one name", N_BANGALORE, "none on total; fixes city splits"),
    ("booking id", f"{N_ID_ROWS:,} rows carry a 21-character id (a stray 0 appended)", "repair: drop last character", "collection app and lab carry the same stray id; no collision with any valid id", N_ID_ROWS,
     f"keeps these bookings (kit rejects them): growth {G['B']:+.2f}% with, {G['C']:+.2f}% without"),
    ("test code", "blank test code on some booking-test rows", "keep, flag", "the test happened; collection app knows 97% of these visits", int(rows.blank_code.sum()), "counted as tests; value unknown"),
    ("created_at", "created_at after slot_at on some bookings (always by 2 days)", "keep, ignore created_at", "sample is collected at slot_at; created_at is not used", int(rows.created_after_slot.sum()), "none"),
    ("reschedules", "nova writes a successor record for a rescheduled visit; collection and lab stay on the original", "keep the original, drop the successor", "one visit, one record; keeps the status the door recorded", len(succ),
     f"tests {G['A']:+.2f}% (rows) -> {G['B']:+.2f}% (visits)"),
    ("collections", "samples scanned twice (exact duplicate rows)", "drop duplicates", "one sample event per booking", N_DOUBLE_SCANS, "none"),
    ("collections", "sample timestamped >1 day before its slot", "keep, flag", "status is right even if the clock is not", N_SKEW, "none"),
    ("period window", "starter cuts a half at midnight on 30 June, dropping the last day's bookings", "half-open windows [1 Jan, 1 Jul)", "30 June is a working day", int(((visits.slot_at >= "2026-06-30") & (visits.slot_at < "2026-07-01")).sum() + ((visits.slot_at >= "2025-06-30") & (visits.slot_at < "2025-07-01")).sum()),
     f"window alone: {G['C']:+.2f}% -> {G['D']:+.2f}%"),
    ("nova bookings", f"{N_EMPTY_TESTS} nova bookings have an empty tests list and would vanish at the grain conversion", "keep as visits with 0 known tests", "the visit exists; only its tests are unknown", N_EMPTY_TESTS, "visits +%d; tests +%d (one empty original takes its successor's tests)" % (N_EMPTY_TESTS, N_INHERIT_TESTS)),
    ("lab results", f"{N_GHOST_RES:,} lab rows carry booking ids that exist in no booking export (serials >= 900000)", "quarantine in story_lab_rows_without_booking.csv", "no visit to attach them to; cannot be placed in a city or quarter", N_GHOST_RES, "excluded from visit-level lab counts; 0.5% of lab rows"),
    ("patients", "same person under several patient ids", "merge on phone + name + dob", "growth is in people, not ids", N_PAT_DUP, "people count falls ~2%; growth barely moves"),
], columns=["table", "what we found", "decision", "reason", "rows affected", "effect on the headline"])
DECISIONS.insert(0, "#", range(1, len(DECISIONS) + 1))
pd.set_option("display.max_colwidth", 80)
display(DECISIONS)
DECISIONS.to_csv(CONFIG["clean"] / "story_decisions.csv", index=False)
visits.to_csv(CONFIG["clean"] / "story_visits.csv.gz", index=False, compression="gzip")
print(f"wrote story_visits.csv.gz ({len(visits):,} visits) and story_decisions.csv   [{time.time()-T_START:.0f}s so far]")

### SUM · Part 1

In [ ]:
print(f"{N_RAW_ROWS:,} raw booking-test rows from two booking systems became {len(visits):,} visits ({int(visits.n_tests.sum()):,} tests booked).")
print(f"Tests booked, H1 2026 vs H1 2025, on our table: {G['B']:+.2f}%.  The starter kit's way gives {G['D']:+.2f}%.")
print("Every row of the difference is in the ledger above and the decisions table; nothing was dropped without a count.")

---
# PART 2 · Brief A — Where did the gap go?

> *"Show me the part of my business that is short, by how much, and where."* — Dr Priya Menon, COO

**The path:** (1) reproduce the 5 and find out what the plan was written in → (2) split growth into **people × visits per person × tests per visit** → (3) split the shortfall by **city** → (4) split by **channel** → (5) hand the cities that look worst to Part 3.

Every table below uses visits in the two halves only (`V`), one row per visit from Part 1.

In [ ]:
V = visits[visits.half != "other"].copy()
V["reached_door"] = V.status.isin(["collected", "no_show"])
print(f"{len(V):,} visits in the two halves: {V.groupby('half').size().to_dict()}")

## 2.1 · Reproduce the 5 percent, and find out what the plan counts

In [ ]:
defs = {
    "tests booked (every visit)":            V.groupby("half").n_tests.sum(),
    "tests on visits that reached the door": V[V.reached_door].groupby("half").n_tests.sum(),
    "tests on collected visits":             V[V.status == "collected"].groupby("half").n_tests.sum(),
    "tests resulted by the lab":             V.groupby("half").tests_resulted.sum(),
}
plan = pd.read_csv(CONFIG["raw"] / "plan" / "plan_2026_H1.csv")
plan["city"] = plan["city"].replace({"Bangalore": "Bengaluru"})
plan_city = plan.set_index("city")[[c for c in plan.columns if c.startswith("2026-")]].sum(axis=1)
PLAN_TOTAL = int(plan_city.sum())
t1 = pd.DataFrame({k: {"H1 2025": int(v["H1_2025"]), "H1 2026": int(v["H1_2026"])} for k, v in defs.items()}).T
t1["growth %"] = pct(t1["H1 2026"], t1["H1 2025"])
t1["growth the plan needs on this base %"] = pct(PLAN_TOTAL, t1["H1 2025"])
display(t1)
BASIS = (t1["growth the plan needs on this base %"] - 18).abs().idxmin()
print(f"Plan, summed over six cities: {PLAN_TOTAL:,} tests.  The definition on which that is closest to +18%: '{BASIS}'.")
TESTS_25, TESTS_26 = int(t1.loc["tests booked (every visit)", "H1 2025"]), int(t1.loc["tests booked (every visit)", "H1 2026"])
G_TESTS = pct(TESTS_26, TESTS_25); G_PLAN = pct(PLAN_TOTAL, TESTS_25); GAP_TESTS = PLAN_TOTAL - TESTS_26
check("T1: we reproduce Dr Menon's 5 percent", abs(G_TESTS - 5.0) < 0.2, f"{G_TESTS:+.2f}%  ({TESTS_26:,} / {TESTS_25:,})")
check("the plan is written in tests booked", BASIS == "tests booked (every visit)")
check("the lab agrees with the bookings in total (independent witness)", abs(t1.loc["tests resulted by the lab", "growth %"] - G_TESTS) < 1.5, f"lab {t1.loc['tests resulted by the lab', 'growth %']:+.2f}% vs booked {G_TESTS:+.2f}%")

### SUM
The 5 reproduces on **tests booked** (every booked test, including visits that never reached the door). That is also the unit the plan was written in, so it is the right one to measure the gap in. The plan asks for a little under +19% on this base (the board's "18") — a shortfall of about 13 to 14 points. We keep the exact numbers, not the rounded ones.

## 2.2 · The tree: people × visits per person × tests per visit
Handbook Step 2: *multiply the branch growth rates back together; if they miss the total, a branch is missing.* A **person** is a de-duplicated patient (Part 1.6), counted once however often they came.

In [ ]:
def tree(df, by=None):
    keys = ([by] if by else []) + ["half"]
    g = df.groupby(keys, observed=True).agg(tests=("n_tests", "sum"), visits=("booking_id", "size"), people=("person_id", "nunique"))
    g["visits_per_person"] = g.visits / g.people
    g["tests_per_visit"] = g.tests / g.visits
    return g

T = tree(V)
a, b = T.loc["H1_2025"], T.loc["H1_2026"]
branches = pd.DataFrame({"H1 2025": a, "H1 2026": b})
branches["growth %"] = pct(branches["H1 2026"], branches["H1 2025"])
branches["log points"] = np.log(branches["H1 2026"] / branches["H1 2025"]) * 100
display(branches.loc[["people", "visits_per_person", "tests_per_visit", "tests"]])
check("branches multiply back to the total (growth factors)", abs((1+branches.loc["people","growth %"]/100) * (1+branches.loc["visits_per_person","growth %"]/100) * (1+branches.loc["tests_per_visit","growth %"]/100) - (1+branches.loc["tests","growth %"]/100)) < 1e-9)
LP = branches["log points"]
check("log points add up: people + visits/person + tests/visit = tests", abs(LP["people"] + LP["visits_per_person"] + LP["tests_per_visit"] - LP["tests"]) < 1e-9,
      f"{LP['people']:+.2f} {LP['visits_per_person']:+.2f} {LP['tests_per_visit']:+.2f} = {LP['tests']:+.2f}")

### Where is the shortfall, branch by branch?
The plan was "set on last year's mix": visits per person and tests per visit held flat, so **people** had to carry all the planned growth. Measure each branch against that assumption, in log points so the pieces add.

In [ ]:
need = np.log(PLAN_TOTAL / TESTS_25) * 100
gap_lp = need - LP["tests"]
short = pd.DataFrame({
    "log points": [need - LP["people"], -LP["visits_per_person"], -LP["tests_per_visit"], gap_lp],
}, index=["people (needed to carry the whole plan, minus what came)", "visits per person (plan: flat; actual below)", "tests per visit (plan: flat; actual above, so it helps)", "= gap to plan"])
short["share of gap %"] = short["log points"] / gap_lp * 100
display(short)
check("branch shortfalls add up to the gap", abs(short["log points"].iloc[:3].sum() - gap_lp) < 1e-9)

### The patients branch, opened: new people and returning people

In [ ]:
start = {k: pd.Timestamp(v[0]) for k, v in CONFIG["halves"].items()}
V["is_new"] = V.signup >= V.half.map(start)
check("every visit has a sign-up date", bool(V.signup.notna().all()), int(V.signup.isna().sum()))
nr = V.groupby(["half", "is_new"]).person_id.nunique().unstack("half")
nr.index = nr.index.map({True: "new (signed up in the half)", False: "returning"})
nr["growth %"] = pct(nr["H1_2026"], nr["H1_2025"])
display(nr)

### SUM

In [ ]:
print(f"Tests booked {G_TESTS:+.2f}% ({TESTS_26:,} / {TESTS_25:,}) against a plan of {G_PLAN:+.2f}% ({PLAN_TOTAL:,}): short by {GAP_TESTS:,} tests.")
print(f"People {branches.loc['people','growth %']:+.1f}%, visits per person {branches.loc['visits_per_person','growth %']:+.1f}%, tests per visit {branches.loc['tests_per_visit','growth %']:+.1f}%.")
print(f"On the plan's own assumption (flat mix) the people branch accounts for {short['share of gap %'].iloc[0]:.0f}% of the gap; visits per person adds {short['share of gap %'].iloc[1]:.0f}%; tests per visit gives back {-short['share of gap %'].iloc[2]:.0f}%.")
print(f"The gap is a PEOPLE problem, not an order-size problem: fewer people came than the plan needed, and they did not come more often.")

## 2.3 · Which cities own the gap?
The plan is by city. Gap by city = plan − actual tests booked. `lab` is the lab's own growth for the same city, a witness the booking system cannot influence.

In [ ]:
Tc = tree(V, "city")
c25, c26 = Tc.xs("H1_2025", level="half"), Tc.xs("H1_2026", level="half")
city = pd.DataFrame({"H1 2025": c25.tests, "H1 2026": c26.tests, "plan": plan_city})
city["growth %"] = pct(city["H1 2026"], city["H1 2025"])
city["plan growth %"] = pct(city["plan"], city["H1 2025"])
city["short by"] = city["plan"] - city["H1 2026"]
city["share of gap %"] = city["short by"] / city["short by"].sum() * 100
city["share of 2025 base %"] = city["H1 2025"] / city["H1 2025"].sum() * 100
city["people %"] = pct(c26.people, c25.people)
city["visits/person %"] = pct(c26.visits_per_person, c25.visits_per_person)
city["tests/visit %"] = pct(c26.tests_per_visit, c25.tests_per_visit)
lab_c = V.groupby(["city", "half"]).tests_resulted.sum().unstack("half")
city["lab tests %"] = pct(lab_c["H1_2026"], lab_c["H1_2025"])
city = city.sort_values("share of gap %", ascending=False)
display(city)
check("city shortfalls add up to the total gap", int(city["short by"].sum()) == GAP_TESTS, f"{int(city['short by'].sum()):,} = {GAP_TESTS:,}")

In [ ]:
mig = CONFIG["migrated"]
SH_MIG, BASE_MIG = city.loc[mig, "share of gap %"].sum(), city.loc[mig, "share of 2025 base %"].sum()
print(f"Chennai + Jaipur: {SH_MIG:.0f}% of the gap on {BASE_MIG:.0f}% of last year's tests.")
print("Gap per 100 tests of last year's volume:", (city['short by'] / city['H1 2025'] * 100).round(1).sort_values(ascending=False).to_dict())

### SUM
No city hit plan. Every city is short, but the shortfall is **not spread evenly**: Chennai and Jaipur own a bigger share of the gap than their share of the business, and Jaipur is the only city that shrank. Part 3 asks whether that is real. First: which *channel* is short? A city-only view hides a mix shift.

## 2.4 · Which channel is short?
The plan has no channel split, so we apply each city's planned uplift to both of its channels equally (an assumption, stated in the caveat). Gap by channel = planned − actual.

In [ ]:
cc = V.groupby(["city", "channel", "half"]).n_tests.sum().unstack("half")
uplift = (plan_city / city["H1 2025"]).reindex(cc.index.get_level_values("city")).to_numpy()
cc["plan"] = cc["H1_2025"] * uplift
cc["growth %"] = pct(cc["H1_2026"], cc["H1_2025"])
cc["short by"] = cc["plan"] - cc["H1_2026"]
display(cc.round(1))
ch = cc.groupby("channel")[["H1_2025", "H1_2026", "plan", "short by"]].sum()
ch["growth %"] = pct(ch["H1_2026"], ch["H1_2025"]); ch["plan growth %"] = pct(ch["plan"], ch["H1_2025"])
display(ch.round(1))
check("channel gaps add up to the total gap", abs(ch["short by"].sum() - GAP_TESTS) < 1, f"{ch['short by'].sum():,.0f} vs {GAP_TESTS:,}")
ppl_ch = V.groupby(["channel", "half"]).person_id.nunique().unstack("half"); ppl_ch["growth %"] = pct(ppl_ch["H1_2026"], ppl_ch["H1_2025"])
print("People by channel:"); display(ppl_ch)

### The picture for slide 4

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.25, 1]})
order = city.sort_values("growth %").index.tolist()
w = 0.38; xs = np.arange(len(order))
cg = cc["growth %"].unstack("channel").loc[order]
ax[0].barh(xs + w/2, cg["clinic"], w, color=PINK, label="clinic channel")
ax[0].barh(xs - w/2, cg["home"], w, color=VIOLET, label="home collection")
ax[0].scatter(city.loc[order, "plan growth %"], xs, marker="|", s=450, color=INK, zorder=5, label="plan (city)")
ax[0].axvline(0, color=INK, lw=1); ax[0].set_xlim(-31, 52)
for i, (c_, h_) in enumerate(zip(cg["clinic"], cg["home"])):
    ax[0].text(c_ + (1 if c_ >= 0 else -1), i + w/2, f"{c_:+.0f}%", va="center", ha="left" if c_ >= 0 else "right", fontsize=8, color=INK)
    ax[0].text(h_ + 1, i - w/2, f"{h_:+.0f}%", va="center", fontsize=8, color=INK)
ax[0].set_yticks(xs); ax[0].set_yticklabels([o + (" *" if o in mig else "") for o in order])
ax[0].set_xlabel("tests booked, H1 2026 vs H1 2025, % (denominator: H1 2025 tests booked in that city and channel)")
ax[0].set_title("Clinic fell everywhere; home grew everywhere", loc="left", fontsize=11, color=INK); ax[0].legend(frameon=False, fontsize=8, loc="lower right", bbox_to_anchor=(1.0, 0.0))
# bridge: plan -> actual
labels = ["Plan", "Clinic channel\nshort", "Home channel\nahead", "Actual"]
vals = [PLAN_TOTAL, -ch.loc["clinic", "short by"], -ch.loc["home", "short by"], TESTS_26]
bottoms = [0, PLAN_TOTAL - ch.loc["clinic", "short by"], PLAN_TOTAL - ch.loc["clinic", "short by"], 0]
heights = [PLAN_TOTAL, ch.loc["clinic", "short by"], -ch.loc["home", "short by"], TESTS_26]
bottoms[1] = PLAN_TOTAL - ch.loc["clinic", "short by"]
cols = [GREY, PINK, VIOLET, INK]
ax[1].bar(labels, heights, bottom=bottoms, color=cols, width=0.6)
ax[1].set_ylim(TESTS_26 * 0.8, PLAN_TOTAL * 1.03)
for i, (h_, b_) in enumerate(zip(heights, bottoms)):
    lbl = f"{PLAN_TOTAL:,}" if i == 0 else f"{TESTS_26:,}" if i == 3 else (f"-{ch.loc['clinic','short by']:,.0f}" if i == 1 else f"+{-ch.loc['home','short by']:,.0f}")
    ax[1].text(i, b_ + h_ + PLAN_TOTAL * 0.004, lbl, ha="center", fontsize=9, color=INK)
ax[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax[1].set_title("Tests booked, H1 2026: plan to actual", loc="left", fontsize=11, color=INK)
ax[1].set_ylabel("tests booked (y-axis starts at 80% of actual)")
fig.text(0.01, 0.005, "* booking system changed 1 April 2026.  Channel plan = each city's uplift applied to both channels (assumption).", fontsize=8, color=GREY)
plt.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(CONFIG["slides"] / "story_A_where_the_gap_is.png", dpi=200); plt.show()

### SUM · Part 2

In [ ]:
CL, HM = ch.loc["clinic"], ch.loc["home"]
print(f"CLAIM A. Tests booked grew {G_TESTS:+.1f}% against a plan of {G_PLAN:+.1f}%. The clinic channel fell {CL['growth %']:.1f}% (plan {CL['plan growth %']:+.1f}%): it is short by {CL['short by']:,.0f} tests.")
print(f"Home collection grew {HM['growth %']:+.1f}% and is {-HM['short by']:,.0f} tests AHEAD of its share of plan. Net gap {GAP_TESTS:,} tests.")
print(f"Clinic-channel people: {ppl_ch.loc['clinic','growth %']:+.1f}%; home people: {ppl_ch.loc['home','growth %']:+.1f}%.")
worst = cc["growth %"].unstack("channel")["clinic"].sort_values().head(3)
print("Weakest clinic channels:", {k: f"{v:+.1f}%" for k, v in worst.items()})
print(f"\nHAND-OFF to B: Chennai and Jaipur own {SH_MIG:.0f}% of the gap on {BASE_MIG:.0f}% of the base, and both changed booking system on 1 April. Is their fall real?")

---
# PART 3 · Brief B — Chennai and Jaipur: is the fall real, or is it counting?

> *"Before I cut targets or people, I need to know whether the fall is real."* — Dr Priya Menon

**Where Part 2 left us:** Chennai and Jaipur own 44% of the gap on 29% of last year's tests, and both moved to the new booking system on 1 April. A booking system that changes how it counts could manufacture a fall. So:

1. **Confirm and compare like with like**: same quarter a year earlier, same unit, and always *against the four cities that did not change* (a city's number means nothing until we know what a healthy city did in the same quarter).
2. **Look through four lenses.** Two are the booking system; two (the collection app and the lab) never touch it, so a migration cannot move them.
3. **Split the Q2 change into counting and real**, and place the real part on a date and on a branch of the tree.
4. **Name what we cannot rule out**, and the one question that would settle it.

The handbook's ladder for this brief has two missing rungs. Ours: **3. remove the counting difference** and **4. compare against a control group**.

In [ ]:
QS = ["2025Q1", "2025Q2", "2026Q1", "2026Q2"]
CITY_CODE = pd.read_csv(CONFIG["raw"] / "reference" / "cities.csv").set_index("city_code")["city"].to_dict()
CTRL = "Other four cities"
MIG = CONFIG["migrated"]
to_grp = lambda s: s.where(s.isin(MIG), CTRL)
GRPS = MIG + [CTRL]

Vq = visits[visits.quarter.isin(QS)].copy()
Vq["grp"] = to_grp(Vq.city)
Vq["reached_door"] = Vq.status.isin(["collected", "no_show"])
Vq["is_new_q"] = Vq.signup >= Vq.slot_at.dt.to_period("Q").dt.start_time      # signed up inside the quarter of the visit
bkq = bk.assign(quarter=bk.slot_at.dt.to_period("Q").astype(str))
bkq = bkq[bkq.quarter.isin(QS)].assign(grp=lambda d: to_grp(d.city))

def by_grp_q(df, value=None):
    g = df.groupby(["grp", "quarter"])
    s = g.size() if value is None else g[value].sum() if value != "people" else g["person_id"].nunique()
    return s.unstack("quarter").reindex(index=GRPS, columns=QS)
yoy = lambda t, q: pct(t[f"2026{q}"], t[f"2025{q}"])

## 3.1 · Four lenses on the same cities
| Lens | What it counts | Can the booking migration move it? |
|---|---|---|
| 1 Booking export | every record the booking systems wrote (a reschedule writes two in nova) | **yes** |
| 2 Visits | one row per visit, our rule from Part 1 | no, if the rule is right |
| 3 Collection app | samples taken plus no-shows, by clinic and the time of collection | **no** |
| 4 Lab | test results posted, by the city code in the booking id and the time of result | **no** |

Lenses 3 and 4 are read straight from their own files, not from the visits table, so they are independent witnesses.

In [ ]:
# lens 3: the collection app, straight from its own file
c3 = coll.assign(quarter=pd.to_datetime(coll.collected_at).dt.to_period("Q").astype(str),
                 city=coll.booking_id.str.extract(r"(?i)(?:BK-|nova_)([A-Za-z]{3})")[0].str.upper().map(CITY_CODE))
c3 = c3[c3.quarter.isin(QS)].assign(grp=lambda d: to_grp(d.city))
# lens 4: the lab, straight from its own file; the phantom-id rows from Part 1.5 are left out
r4 = res[~res.booking_id.isin(orph.booking_id)]
r4 = r4.assign(quarter=pd.to_datetime(r4.result_ts, errors="coerce").dt.to_period("Q").astype(str),
               city=r4.booking_id.str.extract(r"(?i)(?:BK-|nova_)([A-Za-z]{3})")[0].str.upper().map(CITY_CODE))
r4 = r4[r4.quarter.isin(QS)].assign(grp=lambda d: to_grp(d.city))

LENS = {"1 booking export (every record)": by_grp_q(bkq), "2 visits (one per visit)": by_grp_q(Vq),
        "3 collection app (samples + no-shows)": by_grp_q(c3), "4 lab (test results)": by_grp_q(r4)}
KEYS = list(LENS)
check("lens 3 and 4 found a city for every row", bool(c3.city.notna().all() and r4.city.notna().all()))
lens_tab = pd.concat({k: pd.DataFrame({"Q1 vs last year %": yoy(v, "Q1"), "Q2 vs last year %": yoy(v, "Q2")}) for k, v in LENS.items()}, names=["lens"])
for q_ in ("Q1", "Q2"):
    lens_tab[f"{q_} gap to other four, pts"] = lens_tab[f"{q_} vs last year %"] - lens_tab.groupby(level=0)[f"{q_} vs last year %"].transform(lambda s: s.iloc[-1])
display(lens_tab)
print("Each lens has its own level (the lab runs a few points above bookings even before the migration, because fewer visits end as no-shows).")
print("So we compare lenses by their GAP TO THE OTHER FOUR CITIES in the same lens: a common offset cancels.")

### CHECK · do the lenses the migration cannot touch agree with our visits?

In [ ]:
gap = lambda k, q="Q2": lens_tab.loc[k, f"{q} gap to other four, pts"]
v_q2, c_q2, l_q2, r_q2 = (yoy(LENS[k], "Q2") for k in (KEYS[1], KEYS[2], KEYS[3], KEYS[0]))
for g_ in MIG:
    gr, gv, gc, gl = (gap(k)[g_] for k in KEYS)
    check(f"{g_}: visits, collection app and lab put Q2 the same distance below the other four (within 2.5 pts)", max(gv, gc, gl) - min(gv, gc, gl) < 2.5, f"visits {gv:+.1f}, collection {gc:+.1f}, lab {gl:+.1f} pts")
    check(f"{g_}: the raw booking export is the odd one out (its gap is >3 pts smaller than the visits' gap)", gr - gv > 3.0, f"export {gr:+.1f} vs visits {gv:+.1f} pts")

### SUM

In [ ]:
for g_ in MIG:
    print(f"{g_}, Q2 vs Q2 a year earlier: raw export {r_q2[g_]:+.1f}% | visits {v_q2[g_]:+.1f}% | collection app {c_q2[g_]:+.1f}% | lab {l_q2[g_]:+.1f}%.   Gap to the other four (pts): raw {gap(KEYS[0])[g_]:+.1f} | visits {gap(KEYS[1])[g_]:+.1f} | collection {gap(KEYS[2])[g_]:+.1f} | lab {gap(KEYS[3])[g_]:+.1f}")
print(f"{CTRL}: visits {v_q2[CTRL]:+.1f}% | collection app {c_q2[CTRL]:+.1f}% | lab {l_q2[CTRL]:+.1f}%")
print("\nThe two lenses that cannot be touched by a booking migration put each city the same distance below its healthy peers as our visit table does.")
print("The raw export of the NEW system is the only lens that disagrees: it writes two records for a rescheduled visit, so it flatters Q2.")

## 3.2 · Split the Q2 change: counting, market, an older trend, and what moved at the switch
Start from Q2 2025 visits in each city and walk to what the new system exported for Q2 2026. Every step is a count of visits.

| Step | Definition |
|---|---|
| base | Q2 2025 visits |
| market | what the base would have done at the **other four cities'** Q2 growth |
| older trend | the city's Q1 growth gap to those cities, **before** the migration, carried into Q2 |
| change since Q1 | the rest of the change in visits: how the gap to the market moved between Q1 and Q2. Negative = widened at the switch, positive = narrowed |
| counting | extra records the new system exported for the same visits (reschedule duplicates) |

In [ ]:
V_, R_ = LENS[KEYS[1]], LENS[KEYS[0]]
ctrl_q1, ctrl_q2 = yoy(V_, "Q1")[CTRL] / 100, yoy(V_, "Q2")[CTRL] / 100
wf = pd.DataFrame(index=MIG)
wf["base: Q2 2025 visits"] = V_.loc[MIG, "2025Q2"]
wf["market (other four cities' Q2 growth)"] = wf["base: Q2 2025 visits"] * ctrl_q2
wf["older trend (Q1 gap to market, pre-migration)"] = wf["base: Q2 2025 visits"] * (yoy(V_, "Q1")[MIG] / 100 - ctrl_q1)
wf["Q2 2026 visits (our table)"] = V_.loc[MIG, "2026Q2"]
wf["change since Q1 (residual)"] = wf["Q2 2026 visits (our table)"] - wf["base: Q2 2025 visits"] - wf.iloc[:, 1] - wf.iloc[:, 2]
wf["counting (extra records in the new export)"] = R_.loc[MIG, "2026Q2"] - V_.loc[MIG, "2026Q2"]
wf["Q2 2026 as exported by the new system"] = R_.loc[MIG, "2026Q2"]
display(wf.T.round(0))
check("the steps walk from base to the raw export exactly", bool(((wf.iloc[:, 0] + wf.iloc[:, 1] + wf.iloc[:, 2] + wf["change since Q1 (residual)"] + wf["counting (extra records in the new export)"]) - wf["Q2 2026 as exported by the new system"]).abs().max() < 1e-6))
pp = pd.DataFrame({"market": wf.iloc[:, 1], "older trend": wf.iloc[:, 2], "change since Q1": wf["change since Q1 (residual)"], "counting": wf["counting (extra records in the new export)"]}).div(wf.iloc[:, 0], axis=0) * 100
pp["real change (visits)"] = pp[["market", "older trend", "change since Q1"]].sum(axis=1)
pp["= raw export change"] = pp["real change (visits)"] + pp["counting"]
print("The same, in percent of each city's Q2 2025 base:"); display(pp)

### SUM

In [ ]:
for g_ in MIG:
    p_ = pp.loc[g_]
    print(f"{g_}: counting {p_['counting']:+.1f} pts (extra records; hides part of the fall). Real change {p_['real change (visits)']:+.1f}% = market {p_['market']:+.1f} + older trend {p_['older trend']:+.1f} + change since Q1 {p_['change since Q1']:+.1f} (gap to the market {'widened' if p_['change since Q1'] < 0 else 'narrowed'} at the switch).")
print("\nCounting explains none of the fall. It points the wrong way: the new system's raw export looks BETTER than the truth.")

## 3.3 · When did it start?
A real demand shift has a date, and a counting artefact switches on the day the system does (1 April). Compare each city's gap to the other four in Q1 (before) and Q2 (after).

In [ ]:
mon = visits[visits.slot_at < "2026-07-01"].assign(grp=lambda d: to_grp(d.city)).groupby(["grp", "month"]).size().unstack("month").reindex(GRPS)
months26 = [m for m in mon.columns if m.startswith("2026-")]
monthly = pd.DataFrame({m: pct(mon[m], mon[f"2025-{m[-2:]}"]) for m in months26}).T
display(monthly)
step = pd.DataFrame({"Q1 gap to other four, pts (before)": gap(KEYS[1], "Q1")[MIG], "Q2 gap to other four, pts (after)": gap(KEYS[1], "Q2")[MIG]})
step["moved at the switch, pts"] = step.iloc[:, 1] - step.iloc[:, 0]
display(step)

### SUM

In [ ]:
for g_ in MIG:
    s_ = step.loc[g_]
    print(f"{g_}: {s_.iloc[0]:+.1f} pts against the other four before the migration, {s_.iloc[1]:+.1f} after; the gap {'widened' if s_.iloc[2] < 0 else 'narrowed'} by {abs(s_.iloc[2]):.1f} pts at the switch.")
print("\nIn both cities most of the gap to a healthy city was already there in Q1. The migration did not create it.")

## 3.4 · Which branch of the tree, and which patients?
If the fall is real it sits on a branch (people, visits per person, tests per visit) and in a segment. Same Q2-on-Q2 comparison, city against control.

In [ ]:
def tree_q(df):
    d = df.copy(); d["half"] = d["quarter"]            # tree() groups by the 'half' column; here it holds the quarter
    return tree(d, "grp")
Tq = tree_q(Vq[Vq.quarter.isin(["2025Q2", "2026Q2"])])
a_, b_ = Tq.xs("2025Q2", level="half"), Tq.xs("2026Q2", level="half")
br = pd.DataFrame({"visits %": pct(b_.visits, a_.visits), "people %": pct(b_.people, a_.people), "visits per person %": pct(b_.visits_per_person, a_.visits_per_person), "tests per visit %": pct(b_.tests_per_visit, a_.tests_per_visit)}).reindex(GRPS)
display(br)

### The people branch, opened: new people and returning people (Q1 is before the switch, Q2 after)

In [ ]:
new_p = by_grp_q(Vq[Vq.is_new_q], "people"); ret_p = by_grp_q(Vq[~Vq.is_new_q], "people")
# lens 5: the patient master (a separate system again): sign-ups by city and quarter
su = pat.assign(quarter=pat.signup.dt.to_period("Q").astype(str), grp=lambda d: to_grp(d.city))
su = su[su.quarter.isin(QS)].groupby(["grp", "quarter"]).size().unstack("quarter").reindex(index=GRPS, columns=QS)
newret = pd.DataFrame({
    "new people, Q1 %": yoy(new_p, "Q1"), "new people, Q2 %": yoy(new_p, "Q2"),
    "returning people, Q1 %": yoy(ret_p, "Q1"), "returning people, Q2 %": yoy(ret_p, "Q2"),
    "sign-ups in patient master, Q1 %": yoy(su, "Q1"), "sign-ups in patient master, Q2 %": yoy(su, "Q2")}).reindex(GRPS)
display(newret.T)

### Segments: channel, source, and clinic

In [ ]:
Vq2 = Vq[Vq.quarter.isin(["2025Q2", "2026Q2"])]
def seg_yoy(col):
    t = Vq2.groupby(["grp", col, "quarter"]).size().unstack("quarter")
    return pct(t["2026Q2"], t["2025Q2"]).unstack(col).reindex(GRPS)
print("Visits, Q2 vs Q2, by channel and by source:")
display(pd.concat({"channel": seg_yoy("channel"), "source": seg_yoy("source")}, axis=1))
cl = Vq[Vq.grp.isin(MIG)].groupby(["city", "clinic_id", "quarter"]).size().unstack("quarter")
cl = cl[cl["2025Q2"] >= 500]
cl_t = pd.DataFrame({"Q1 vs last year %": pct(cl["2026Q1"], cl["2025Q1"]), "Q2 vs last year %": pct(cl["2026Q2"], cl["2025Q2"])})
display(cl_t)
N_CLIN_FALL, N_CLIN = int((cl_t["Q2 vs last year %"] < 0).sum()), len(cl_t)
print(f"clinics in these two cities with a Q2 fall: {N_CLIN_FALL} of {N_CLIN} (JAI-05 opened in Feb 2026 and is excluded as too small)")

### SUM

In [ ]:
for g_ in MIG:
    r_ = br.loc[g_]
    print(f"{g_}: visits {r_['visits %']:+.1f}% = people {r_['people %']:+.1f}%, visits per person {r_['visits per person %']:+.1f}%, tests per visit {r_['tests per visit %']:+.1f}%.   New people {newret.loc[g_, 'new people, Q2 %']:+.1f}% (Q1 {newret.loc[g_, 'new people, Q1 %']:+.1f}%), returning {newret.loc[g_, 'returning people, Q2 %']:+.1f}%.")
print(f"Other four: people {br.loc[CTRL, 'people %']:+.1f}%, new people {newret.loc[CTRL, 'new people, Q2 %']:+.1f}%, returning {newret.loc[CTRL, 'returning people, Q2 %']:+.1f}%.")
print(f"\nThe fall is a PEOPLE fall, and almost all of it is NEW patients; returning patients are about flat. It is spread across the clinics ({N_CLIN_FALL} of {N_CLIN} fell), not one site.")

## 3.5 · The picture, and what we cannot rule out

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.15, 1]})
mm = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
for g_, col, ls in [("Chennai", PINK, "-"), ("Jaipur", VIOLET, "-"), (CTRL, GREY, "--")]:
    ax[0].plot(mm, monthly[g_].to_numpy(), color=col, lw=2.4, ls=ls, marker="o", label=g_)
    ax[0].text(5.12, monthly[g_].iloc[-1] + {"Chennai": -0.9, "Jaipur": 0.9}.get(g_, 0), f"{monthly[g_].iloc[-1]:+.0f}%", color=col, va="center", fontsize=9)
ax[0].axvline(2.5, color=INK, lw=1, ls=":"); ax[0].text(2.58, ax[0].get_ylim()[1], "new booking system\nin Chennai and Jaipur", fontsize=8, color=INK, va="top")
ax[0].axhline(0, color=INK, lw=0.8); ax[0].set_xlim(-0.2, 5.7)
ax[0].set_ylabel("visits vs same month last year, %"); ax[0].set_title("Monthly visits against a year earlier", loc="left", fontsize=11, color=INK)
ax[0].legend(frameon=False, fontsize=8, loc="lower left")
lab_names = ["booking export\n(raw)", "visits\n(ours)", "collection\napp", "lab"]
x = np.arange(len(lab_names)); w = 0.26
for i, (g_, col) in enumerate([("Chennai", PINK), ("Jaipur", VIOLET), (CTRL, GREY)]):
    vals_ = [yoy(LENS[k], "Q2")[g_] for k in KEYS]
    ax[1].bar(x + (i - 1) * w, vals_, w, color=col, label=g_)
    for xi, v_ in zip(x + (i - 1) * w, vals_): ax[1].text(xi, v_ + (0.4 if v_ >= 0 else -0.4), f"{v_:+.0f}", ha="center", va="bottom" if v_ >= 0 else "top", fontsize=8, color=INK)
ax[1].set_xticks(x); ax[1].set_xticklabels(lab_names, fontsize=9); ax[1].axhline(0, color=INK, lw=0.8)
ax[1].set_ylabel("Q2 2026 vs Q2 2025, %"); ax[1].set_title("Four lenses on Q2: only the raw export is out of line", loc="left", fontsize=11, color=INK); ax[1].legend(frameon=False, fontsize=8)
fig.text(0.01, 0.005, "Denominator: Q2 2025 count in the same lens and city group. 'Other four' = Ahmedabad, Bengaluru, Hyderabad, Pune pooled.", fontsize=8, color=GREY)
plt.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(CONFIG["slides"] / "story_B_real_or_counting.png", dpi=200); plt.show()

### What would make us wrong (handbook, Step 4)
Two explanations sit **outside** the data pack and we cannot rule them out:

1. **Competitor entry (Thyrocare, as the city heads say).** If true we would expect new patients and walk-ins to take the hit, the fall to be uneven across clinics (worse near the new centres) and a date that matches the openings. The segment tables above show the first; we do not have the openings or the clinic locations to test the other two.
2. **A change in how new patients are registered or booked** (an app or phone-line step that now loses people before they complete a booking). Such a loss would not be a counting difference: no booking is written, so the collection app and the lab would not see it either. The patient master is a separate system and also shows the fall in sign-ups, which weakens this explanation but does not remove it.

**The one question that settles the most:** *"On what dates did Thyrocare open in each city, and near which of our clinics?"* One answer separates (1) from (2) and tells us which clinics to visit first.

### SUM · Part 3

In [ ]:
c_, j_ = pp.loc["Chennai"], pp.loc["Jaipur"]
print(f"CLAIM B. The fall is real, and counting is not the reason. The new system's raw export overstates Q2 by {c_['counting']:.1f} pts in Chennai and {j_['counting']:.1f} in Jaipur (reschedule duplicates), so the true picture is worse than the export shows.")
print(f"Against the four unchanged cities, Chennai is {gap(KEYS[1])['Chennai']:+.1f} pts and Jaipur {gap(KEYS[1])['Jaipur']:+.1f} pts in Q2 (collection app {gap(KEYS[2])['Chennai']:+.1f} / {gap(KEYS[2])['Jaipur']:+.1f}; lab {gap(KEYS[3])['Chennai']:+.1f} / {gap(KEYS[3])['Jaipur']:+.1f}).")
print(f"Most of that gap predates the switch (Q1: Chennai {step.loc['Chennai'].iloc[0]:+.1f}, Jaipur {step.loc['Jaipur'].iloc[0]:+.1f} pts). The switch moved Chennai's gap by {step.loc['Chennai'].iloc[2]:+.1f} pts and Jaipur's by {step.loc['Jaipur'].iloc[2]:+.1f} pts.")
print("It is a new-patient fall, spread across clinics. The cause is outside this data: ask the city heads for the Thyrocare opening dates and sites.")
print(f"\nHAND-OFF to D: Chennai is where the two problems meet. CHE-03 is in Chennai. Is its no-show rate a symptom of this same fall, or a separate problem?")

---
# PART 4 · Brief D — Is that clinic's no-show rate really worse?

> *"Tell me whether the difference is real, and if it is, whom it affects."* — Dr Priya Menon

**Where Part 3 left us:** Chennai's fall is real and is mostly a new-patient shortfall. Separately, one Chennai clinic looks bad on no-shows. Is it the same problem, or a different one?

The method (handbook, brief D): **define** a no-show once and apply it to every clinic with its denominator → **chance reference** (how often would a clinic this size look this bad by luck?) → **like with like** (same slots, same booking routes, same channel) → **verdict**: real, noise, or not yet knowable → **price** being wrong, and put it next to the gap from Part 2.

## 4.1 · Define it once, show every rate with its count

In [ ]:
D = visits[visits.status.isin(["collected", "no_show"]) & (visits.slot_at >= "2025-01-01") & (visits.slot_at < "2026-07-01")].copy()
D["ns"] = (D.status == "no_show").astype(int)
hr = D.slot_at.dt.hour
D["slot_band"] = np.select([hr < 10, hr < 14], ["morning", "midday"], "evening")
D["load"] = np.select([D.n_tests <= 1, D.n_tests <= 3], ["1 test", "2-3 tests"], "4+ tests")
STRATA = ["channel", "slot_band", "source", "load"]
N_ALL, K_ALL = len(D), int(D.ns.sum()); P_NET = K_ALL / N_ALL
print("DEFINITION. A no-show is a visit the collection app marked 'no_show' (the phlebotomist was there and the patient was not).")
print("DENOMINATOR. Visits that reached the collection app (collected + no-show), Jan 2025 to Jun 2026.")
print(f"Bookings with no collection row ({int((visits.status == 'no_collection_record').sum()):,}) are neither: cancelled or unknown, spread evenly across cities (Part 1.4). They stay out of numerator and denominator.")
print(f"\nNetwork: {K_ALL:,} no-shows / {N_ALL:,} visits = {P_NET*100:.2f}%")

rate = D.groupby("clinic_id").agg(visits=("ns", "size"), no_shows=("ns", "sum"), tests_per_visit=("n_tests", "mean"))
rate["rate %"] = rate.no_shows / rate.visits * 100
rate["vs network, pts"] = rate["rate %"] - P_NET * 100
rate["size rank"] = rate.visits.rank(ascending=False).astype(int)
print("\nThe league table a regional head would read (ranked by raw rate):")
display(rate.sort_values("rate %", ascending=False).head(5))

### Two clinics, two different questions
The top of the league table is **not** the clinic with the problem the manager was accused of. Rank 1 is a clinic with a handful of visits; rank 2 is a large clinic. They need different answers, so name them.

In [ ]:
se = lambda n: np.sqrt(P_NET * (1 - P_NET) / n)
rate["z (raw)"] = (rate["rate %"] / 100 - P_NET) / se(rate.visits)
FOCUS = rate["z (raw)"].idxmax()          # the clinic furthest from the network once its size is taken into account
SMALL = rate.visits.idxmin()              # the smallest clinic
n_f, k_f = int(rate.loc[FOCUS, "visits"]), int(rate.loc[FOCUS, "no_shows"])
print(f"League table rank 1 is {rate['rate %'].idxmax()} ({rate['rate %'].max():.1f}% on {int(rate.loc[rate['rate %'].idxmax(), 'visits'])} visits).  The clinic furthest from the network once size is allowed for is {FOCUS}.")
print(f"FOCUS = {FOCUS}: {k_f:,} no-shows / {n_f:,} visits = {rate.loc[FOCUS, 'rate %']:.2f}%  (size rank {rate.loc[FOCUS, 'size rank']} of {len(rate)}, z = {rate.loc[FOCUS, 'z (raw)']:+.1f} SE).")
print(f"SMALL = {SMALL}: {int(rate.loc[SMALL, 'no_shows'])} no-shows / {int(rate.loc[SMALL, 'visits'])} visits = {rate.loc[SMALL, 'rate %']:.1f}%.")
alt = visits[(visits.slot_at >= "2025-01-01") & (visits.slot_at < "2026-07-01")].copy()
alt["ns_alt"] = alt.status.isin(["no_show", "no_collection_record"]).astype(int)       # the handbook's wording read literally: 'no sample was ever collected'
alt_r = alt.groupby("clinic_id").agg(n=("ns_alt", "size"), r=("ns_alt", "mean"))
alt_z = (alt_r.r - alt.ns_alt.mean()) / np.sqrt(alt.ns_alt.mean() * (1 - alt.ns_alt.mean()) / alt_r.n)
check("the same clinic stands out under the rejected, literal definition (includes cancelled/unknown)", alt_z.idxmax() == FOCUS, f"{alt_z.idxmax()} at {alt_z.max():+.1f} SE")

### SUM
A no-show is what the door recorded, with the denominator printed beside every rate. The literal reading ("no sample collected") also contains cancellations the patient gave notice of, which cost the lab nothing; both readings single out the same clinic, so the choice does not decide the answer.

## 4.2 · The chance reference: how often would a clinic this size look this bad by luck alone?
Deal the network's no-shows at random to a clinic of the same size, 20,000 times. For a 0/1 outcome and one clinic of fixed size that is a hypergeometric draw, which NumPy does in one call (the slow loop version is in notebook 05). The result is the band that luck alone produces.

In [ ]:
def luck(n, k_total=K_ALL, n_total=N_ALL, draws=20000):
    d = rng.hypergeometric(k_total, n_total - k_total, n, size=draws) / n
    return d, np.percentile(d, [2.5, 97.5])
for name in (FOCUS, SMALL):
    n_, k_ = int(rate.loc[name, "visits"]), int(rate.loc[name, "no_shows"])
    d_, (lo_, hi_) = luck(n_)
    p_ = float((d_ >= k_ / n_).mean())
    print(f"{name}: real {k_/n_*100:.2f}% on {n_:,} visits. Luck alone: {d_.mean()*100:.2f}%, 95% band {lo_*100:.2f}% to {hi_*100:.2f}%. Shuffled worlds at least this bad: {p_:.5f}" + ("  (none of 20,000)" if p_ == 0 else ""))
d_f, (lo_f, hi_f) = luck(n_f)
P_SHUFFLE_FOCUS = float((d_f >= k_f / n_f).mean())
outside = rate[rate["z (raw)"].abs() > 3.09]
print("\nClinics outside the 99.8% funnel on the raw rate:", {k: f"{v:+.1f} SE" for k, v in outside["z (raw)"].round(1).items()})

### What the shuffle assumes
It assumes a booking is equally likely to be a no-show at any clinic. That fails if clinics serve different mixes. So the shuffle says the raw gap at the large clinic is **not luck**, and says nothing yet about *why*. Next: hold the mix fixed.

## 4.3 · Like with like: same channel, same slot, same booking route, same number of tests
First, how different is the mix? Then, for every clinic, build the no-show rate it **should** have if it behaved like the rest of the network in each matching cell (channel x slot band x booking source x tests per visit). The expected rate in a cell uses the **other thirty clinics only**.

In [ ]:
mix = D.assign(evening=D.slot_band == "evening", phone=D.source == "phone", home=D.channel == "home", walkin=D.source == "walkin").groupby("clinic_id")[["evening", "phone", "home", "walkin"]].mean() * 100
mix_tab = pd.DataFrame({FOCUS: mix.loc[FOCUS], "rest of network": D[D.clinic_id != FOCUS].assign(evening=lambda d: d.slot_band == "evening", phone=lambda d: d.source == "phone", home=lambda d: d.channel == "home", walkin=lambda d: d.source == "walkin")[["evening", "phone", "home", "walkin"]].mean() * 100})
mix_tab.index = ["% evening slots", "% phone bookings", "% home collection", "% walk-in bookings"]
display(mix_tab)
rate_by = pd.concat({"evening": D.groupby("slot_band").ns.mean().rename("no-show %") * 100, "booking route": D.groupby("source").ns.mean().rename("no-show %") * 100, "channel": D.groupby("channel").ns.mean().rename("no-show %") * 100})
print("Network no-show rate by group (the reason mix matters):"); display(rate_by.to_frame())

In [ ]:
def adjusted(frame, strata, by="clinic_id"):
    cell = frame.groupby([by] + strata, observed=True).ns.agg(n="size", k="sum").reset_index()
    tot = cell.groupby(strata, observed=True)[["n", "k"]].sum().rename(columns={"n": "N", "k": "K"}).reset_index()
    m = cell.merge(tot, on=strata, validate="many_to_one")
    m = m[m.N > m.n]                                           # a cell with no other clinic has no comparison
    m["p_other"] = (m.K - m.k) / (m.N - m.n)
    m["exp"] = m.n * m.p_other; m["var"] = m.n * m.p_other * (1 - m.p_other)
    out = m.groupby(by).agg(visits=("n", "sum"), no_shows=("k", "sum"), expected=("exp", "sum"), var=("var", "sum"))
    out["rate %"] = out.no_shows / out.visits * 100
    out["expected %"] = out.expected / out.visits * 100
    out["excess pts"] = out["rate %"] - out["expected %"]
    out["se pts"] = np.sqrt(out["var"]) / out.visits * 100
    out["z"] = (out.no_shows - out.expected) / np.sqrt(out["var"])
    return out

adj = adjusted(D, STRATA)
check("every visit sits in a cell with a comparison group", int(adj.visits.sum()) >= int(0.999 * len(D)), f"{int(adj.visits.sum()):,} of {len(D):,}")
adj["mix effect, pts"] = adj["expected %"] - P_NET * 100          # what its mix alone would add or subtract
adj["raw gap, pts"] = adj["rate %"] - P_NET * 100
a_f = adj.loc[FOCUS]
share_mix = a_f["mix effect, pts"] / a_f["raw gap, pts"] * 100
display(adj.loc[[FOCUS, SMALL], ["visits", "no_shows", "rate %", "raw gap, pts", "mix effect, pts", "excess pts", "se pts", "z"]])
print(f"{FOCUS}: raw gap {a_f['raw gap, pts']:+.2f} pts = mix effect {a_f['mix effect, pts']:+.2f} + clinic effect {a_f['excess pts']:+.2f} (+/-{1.96*a_f['se pts']:.2f} at 95%).  The mix explains {share_mix:.0f}% of the gap.")
out_adj = adj[adj.z.abs() > 3.09]
print("Clinics outside the 99.8% limits after matching the mix:", {k: f"{v:+.1f} SE" for k, v in out_adj.z.round(1).items()} or "none")
check("after matching the mix, no clinic is outside luck's 99.8% limits", out_adj.empty)
check(f"before matching, {FOCUS} was far outside it", abs(rate.loc[FOCUS, 'z (raw)']) > 3.09, f"{rate.loc[FOCUS, 'z (raw)']:+.1f} SE")

### Is the mix a standing feature of the clinic, or something the migration changed?

In [ ]:
qm = D.assign(evening=D.slot_band == "evening", phone=D.source == "phone")
qm["is_focus"] = qm.clinic_id == FOCUS
q_mix = qm.groupby(["quarter", "is_focus"])[["evening", "phone"]].mean().unstack("is_focus") * 100
q_mix.columns = [f"% {a} - {'%s' % FOCUS if b else 'rest of network'}" for a, b in q_mix.columns]
display(q_mix)
oth = D[D.clinic_id != FOCUS].groupby(STRATA, observed=True).ns.mean().rename("p_exp")
cf = D[D.clinic_id == FOCUS].merge(oth, on=STRATA, how="left", validate="many_to_one")
qt = cf.groupby("quarter").agg(visits=("ns", "size"), observed=("ns", "mean"), expected=("p_exp", "mean"), var=("p_exp", lambda s: (s * (1 - s)).sum()))
qt["raw gap, pts"] = (qt.observed - P_NET) * 100
qt["excess for its mix, pts"] = (qt.observed - qt.expected) * 100
qt["95% half-width, pts"] = 1.96 * np.sqrt(qt["var"]) / qt.visits * 100
qt[["observed", "expected"]] *= 100
display(qt[["visits", "observed", "expected", "raw gap, pts", "excess for its mix, pts", "95% half-width, pts"]])
N_Q_RAW = int(((qt["raw gap, pts"] - qt["95% half-width, pts"]) > 0).sum())
check("the raw gap is outside luck's band in every quarter (it is a stable fact, not one bad quarter)", N_Q_RAW == len(qt), f"{N_Q_RAW} of {len(qt)}")
ev = qm.groupby(["quarter", "is_focus"]).evening.mean().unstack("is_focus")
check("the evening share is at least 1.5x the rest of the network in every quarter, before and after the switch", bool((ev[True] > 1.5 * ev[False]).all()), (ev[True] / ev[False]).round(2).to_dict())

### Is it the whole city, or this clinic?

In [ ]:
chen = adj.loc[[c for c in adj.index if c.startswith("CHE")], ["visits", "rate %", "expected %", "excess pts", "z"]]
display(chen)
check("every Chennai clinic is within luck of its expected rate once the mix is matched", bool((chen.z.abs() < 3.09).all()), chen.z.round(1).to_dict())

### And a clinic we cannot judge: how many visits would settle it?

In [ ]:
def visits_needed(p, d_pts, z_a=1.96, z_b=0.84):
    # one clinic against a very large network: Var = p(1-p)/n; detect a difference of d_pts with 80% power at the 5% level
    return int(np.ceil((z_a + z_b) ** 2 * p * (1 - p) / (d_pts / 100) ** 2))
s_ = rate.loc[SMALL]
mde = (1.96 + 0.84) * np.sqrt(P_NET * (1 - P_NET) / s_.visits) * 100
NEED3 = visits_needed(P_NET, 3.0)
NEED_RES = visits_needed(P_NET, abs(a_f["excess pts"]))
print(f"{SMALL} (opened 15 Feb 2026): {int(s_.no_shows)} / {int(s_.visits)} = {s_['rate %']:.1f}%. At that size luck alone can hide or fake a difference of up to {mde:.0f} pts. Verdict: not yet knowable.")
print(f"To settle a 3-point difference at 80% power: about {NEED3:,} visits at the clinic. To settle the {a_f['excess pts']:+.2f}-pt residual at {FOCUS}: about {NEED_RES:,} visits; it has {n_f:,}.")

### The picture for slide 4

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.15, 1]})
ng = np.linspace(rate.visits.min() * 0.7, rate.visits.max() * 1.05, 300)
for z_, a_, lbl in [(1.96, 0.18, "95% luck band"), (3.09, 0.10, "99.8% luck band")]:
    ax[0].fill_between(ng, (P_NET - z_ * se(ng)) * 100, (P_NET + z_ * se(ng)) * 100, color=VIOLET, alpha=a_, lw=0, label=lbl)
ax[0].axhline(P_NET * 100, color=INK, lw=1)
rest = rate.drop([FOCUS, SMALL])
ax[0].scatter(rest.visits, rest["rate %"], s=28, color=GREY, zorder=3, label="other clinics")
ax[0].scatter(rate.loc[FOCUS, "visits"], rate.loc[FOCUS, "rate %"], s=90, color=PINK, zorder=4)
ax[0].annotate(f"{FOCUS}  {rate.loc[FOCUS,'rate %']:.1f}%\n{k_f:,} / {n_f:,}\nraw: far outside luck", (rate.loc[FOCUS, "visits"], rate.loc[FOCUS, "rate %"]), xytext=(14, -4), textcoords="offset points", fontsize=8.5, color=PINK)
ax[0].scatter(rate.loc[SMALL, "visits"], rate.loc[SMALL, "rate %"], s=60, color=GOLD, zorder=4)
ax[0].annotate(f"{SMALL}  {rate.loc[SMALL,'rate %']:.0f}%  ({int(s_.no_shows)} / {int(s_.visits)})\nnot yet knowable", (rate.loc[SMALL, "visits"], rate.loc[SMALL, "rate %"]), xytext=(10, 10), textcoords="offset points", fontsize=8, color=INK)
ax[0].set_xlabel("visits at the clinic, Jan 2025 to Jun 2026 (denominator)"); ax[0].set_ylabel("no-show rate, %")
ax[0].set_title("Every clinic against what luck alone allows (raw rate)", loc="left", fontsize=11, color=INK); ax[0].legend(frameon=False, fontsize=8, loc="upper right")
qx = np.arange(len(qt))
ax[1].axhline(P_NET * 100, color=GREY, lw=1.4, ls=":"); ax[1].text(0, P_NET * 100 + 0.15, "network average", fontsize=8, color=GREY)
ax[1].plot(qx, qt.observed, color=PINK, lw=2.4, marker="o", label=f"{FOCUS} observed")
ax[1].plot(qx, qt.expected, color=VIOLET, lw=2.2, ls="--", marker="o", label=f"expected for {FOCUS}'s mix")
ax[1].fill_between(qx, qt.expected - qt["95% half-width, pts"], qt.expected + qt["95% half-width, pts"], color=VIOLET, alpha=0.15, lw=0, label="95% luck band")
ax[1].set_xticks(qx); ax[1].set_xticklabels(qt.index, fontsize=8); ax[1].set_ylabel("no-show rate, %")
ax[1].set_title("Its own mix predicts its rate, quarter after quarter", loc="left", fontsize=11, color=INK); ax[1].legend(frameon=False, fontsize=8, loc="center left")
fig.text(0.01, 0.005, f"Expected = no-show rate of the other 30 clinics in the same channel, slot band, booking route and tests-per-visit cell. {FOCUS}: {mix_tab.iloc[0,0]:.0f}% evening slots vs {mix_tab.iloc[0,1]:.0f}% elsewhere.", fontsize=8, color=GREY)
plt.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(CONFIG["slides"] / "story_D_one_clinic.png", dpi=200); plt.show()

## 4.4 · What it costs, and how big that is next to the gap from Part 2

In [ ]:
h26 = D[D.half == "H1_2026"]
adj26 = adjusted(h26, STRATA)
tpv = D.loc[D.clinic_id == FOCUS, "n_tests"].mean(); val = D.loc[D.clinic_id == FOCUS, "list_value"].mean()
v26 = int(adj26.loc[FOCUS, "visits"]); net26 = h26.ns.mean()
mix_ns26 = (adj26.loc[FOCUS, "expected"] - net26 * v26)           # extra no-shows its schedule mix brings
clinic_ns26 = adj26.loc[FOCUS, "no_shows"] - adj26.loc[FOCUS, "expected"]
for lbl, ex in [("from its schedule mix (evening and phone heavy)", mix_ns26), ("clinic-specific, after matching the mix", clinic_ns26)]:
    print(f"H1 2026 at {FOCUS}, extra no-shows {lbl}: {ex:,.0f}  ->  {ex*tpv:,.0f} tests, Rs {ex*val:,.0f} of list-price tests, {ex*tpv/GAP_TESTS*100:.2f}% of the plan gap.")
SHARE_GAP_D = (mix_ns26 + max(clinic_ns26, 0)) * tpv / GAP_TESTS * 100
check("D is small next to A (under 2% of the gap), so it must not be sold as the explanation of the 13 points", SHARE_GAP_D < 2, f"{SHARE_GAP_D:.2f}%")
net_ns = (D[D.half.isin(["H1_2025", "H1_2026"])].groupby("half").ns.mean() * 100)
print(f"\nNetwork no-show rate, H1 2025 -> H1 2026: {net_ns['H1_2025']:.2f}% -> {net_ns['H1_2026']:.2f}% (it is not rising across the company).")

### SUM · Part 4

In [ ]:
print(f"CLAIM D. The difference is REAL as a rate and is NOT a clinic-quality problem. {FOCUS}: {rate.loc[FOCUS,'rate %']:.1f}% ({k_f:,} / {n_f:,}) against {P_NET*100:.1f}% for the network ({K_ALL:,} / {N_ALL:,}), Jan 2025 to Jun 2026.")
print(f"Not luck: {'none' if P_SHUFFLE_FOCUS == 0 else format(P_SHUFFLE_FOCUS, '.4f')} of 20,000 shuffled worlds were as bad, and it is {rate.loc[FOCUS,'z (raw)']:+.1f} SE out. Not size: it is rank {rate.loc[FOCUS,'size rank']} of {len(rate)}.")
print(f"But {mix_tab.iloc[0,0]:.0f}% of its visits are evening slots (rest of the network {mix_tab.iloc[0,1]:.0f}%) and {mix_tab.iloc[1,0]:.0f}% are phone bookings (rest {mix_tab.iloc[1,1]:.0f}%), and those groups no-show more everywhere. Matched on channel, slot, route and tests per visit, the clinic effect is {a_f['excess pts']:+.2f} pts (95% band +/-{1.96*a_f['se pts']:.2f}); the mix explains {share_mix:.0f}% of the gap.")
print(f"The same holds in every quarter, before and after the switch, and for every other Chennai clinic. Cost: {mix_ns26:,.0f} extra no-shows in H1 2026 from its mix (~{mix_ns26*tpv:,.0f} tests, {mix_ns26*tpv/GAP_TESTS*100:.2f}% of the plan gap).")
print(f"Not yet knowable: {SMALL} ({int(s_.visits)} visits; luck alone can fake a difference of up to {mde:.0f} pts).")

---
# PART 5 · The board sentence

Claim → evidence → caveat → action, for each level of the story, with the denominator beside every number. This is the handbook's answer shape and the order of the six slides: the question (Dr Menon's words), the picture (the three charts in `slides/`), the claim, the evidence, the caveat and alternative, the action with its cost.

In [ ]:
# what is at stake in B: visits per quarter the two cities would add if they grew like the other four
base_q2 = LENS[KEYS[1]]["2025Q2"]
stake_visits = {g_: -gap(KEYS[1])[g_] / 100 * base_q2[g_] for g_ in MIG}
tpv_q2 = Vq[Vq.quarter == "2026Q2"].groupby("city").n_tests.sum() / Vq[Vq.quarter == "2026Q2"].groupby("city").size()
stake_tests = {g_: stake_visits[g_] * tpv_q2[g_] for g_ in MIG}

A_SHORT = (f"Tests booked grew {G_TESTS:.1f}% ({TESTS_26:,} / {TESTS_25:,}) against a plan of {G_PLAN:.1f}% ({PLAN_TOTAL:,}), H1 2026 vs H1 2025. The whole shortfall of {GAP_TESTS:,} tests "
           f"sits in the clinic channel (tests {CL['growth %']:+.1f}%, {ppl_ch.loc['clinic','growth %']:+.1f}% people); home collection grew {HM['growth %']:+.1f}% and ran {-HM['short by']:,.0f} tests ahead.")
B_SHORT = (f"Chennai and Jaipur own {SH_MIG:.0f}% of the gap on {BASE_MIG:.0f}% of the volume. Their Q2 fall is real, not a counting effect of the new booking system: the collection app and the lab, which the migration cannot touch, "
           f"put them {-gap(KEYS[2])['Chennai']:.0f} and {-gap(KEYS[2])['Jaipur']:.0f} points (collection app) below the four unchanged cities; the new system's raw export overstates Q2 by {pp.loc['Chennai','counting']:.1f} and {pp.loc['Jaipur','counting']:.1f} points. "
           f"It is a new-patient fall (new people {newret.loc['Chennai','new people, Q2 %']:+.0f}% and {newret.loc['Jaipur','new people, Q2 %']:+.0f}% against {newret.loc[CTRL,'new people, Q2 %']:+.0f}% elsewhere), already visible in Q1.")
D_SHORT = (f"{FOCUS}'s no-show rate is {rate.loc[FOCUS,'rate %']:.1f}% ({k_f:,} / {n_f:,}) against {P_NET*100:.1f}% ({K_ALL:,} / {N_ALL:,}): not luck, but {share_mix:.0f}% of the gap is its schedule "
           f"({mix_tab.iloc[0,0]:.0f}% evening slots, {mix_tab.iloc[1,0]:.0f}% phone bookings, against {mix_tab.iloc[0,1]:.0f}% and {mix_tab.iloc[1,1]:.0f}% elsewhere); like with like it is {a_f['excess pts']:+.2f} pts (+/-{1.96*a_f['se pts']:.2f}). Worth {SHARE_GAP_D:.1f}% of the plan gap.")

CLAIMS = pd.DataFrame([
    ("A · Growth", A_SHORT, f"{G_TESTS:+.1f}%", TESTS_26, TESTS_25, "H1 2026 vs H1 2025, six cities, tests booked",
     "The plan has no channel split: we apply each city's planned uplift to both its channels equally. 'People' are de-duplicated by phone + name + date of birth.",
     f"Re-plan by channel. The clinic channel needs its footfall back ({CL['short by']:,.0f} tests short); home growth is real but its extra cost per visit is not in this pack, so do not treat it as a free substitute."),
    ("B · Two cities", B_SHORT, f"{gap(KEYS[2])['Chennai']:+.1f} / {gap(KEYS[2])['Jaipur']:+.1f} pts", int(V_.loc['Chennai', '2026Q2'] + V_.loc['Jaipur', '2026Q2']), int(V_.loc['Chennai', '2025Q2'] + V_.loc['Jaipur', '2025Q2']), "Q2 2026 vs Q2 2025, visits, gap to the other four cities",
     "The cause (Thyrocare, or a registration/booking step that loses new patients) is outside the data. A 2-4 point difference between lenses is normal; the conclusion does not rest on it.",
     f"Do not cut headcount on a counting rationale: there is none. Reset the two cities' targets to a market-adjusted level, hold headcount until the Thyrocare opening dates and sites are known, and point effort at new-patient acquisition. At stake: about {stake_visits['Chennai']:,.0f} (Chennai) and {stake_visits['Jaipur']:,.0f} (Jaipur) visits a quarter, roughly {stake_tests['Chennai']:,.0f} and {stake_tests['Jaipur']:,.0f} tests."),
    ("D · One clinic", D_SHORT, f"{rate.loc[FOCUS,'rate %']:.1f}% vs {P_NET*100:.1f}%", k_f, n_f, f"{FOCUS}, Jan 2025 to Jun 2026, visits that reached the collection app",
     f"Matching uses four things we can see (channel, slot, route, tests per visit). Unseen differences cannot be excluded; the residual {a_f['excess pts']:+.2f} pts would need about {NEED_RES:,} visits to settle.",
     f"Do not replace the manager on this evidence. If the evening-and-phone mix is a choice, test reminder calls on evening phone bookings at {FOCUS}. {SMALL} ({int(s_.visits)} visits) is not yet knowable: look again at about {NEED3:,} visits."),
], columns=["brief", "claim", "number", "numerator", "denominator", "period and unit", "caveat", "action"])
CLAIMS.to_csv(CONFIG["clean"] / "story_claims.csv", index=False)
pd.set_option("display.max_colwidth", 400)
for _, r in CLAIMS.iterrows():
    print(f"[{r.brief}]  {r['number']}   ({r.numerator:,} / {r.denominator:,}; {r['period and unit']})\n  CLAIM   {r.claim}\n  CAVEAT  {r.caveat}\n  ACTION  {r.action}\n")

## The one sentence

In [ ]:
BOARD = (f"Kalpa Health grew tests {G_TESTS:.1f}% against a plan of {G_PLAN:.1f}% because fewer people came to our clinics: the clinic channel is down {-CL['growth %']:.1f}% while home collection is up {HM['growth %']:.1f}%, "
         f"and Chennai and Jaipur, {BASE_MIG:.0f}% of the business, account for {SH_MIG:.0f}% of the shortfall; that fall is real, not a counting effect of the new booking system.")
print(BOARD)
print(f"\nAnd the two follow-ups she will be asked: (D) one Chennai clinic's high no-show rate is real but is its evening-and-phone schedule, worth {SHARE_GAP_D:.1f}% of the gap; (B) the cause of the two cities' new-patient fall is outside our data: ask for the competitor opening dates.")

## The cold run

In [ ]:
ELAPSED = time.time() - T_START
HEADLINE = {"number": round(G_TESTS, 2), "numerator": TESTS_26, "denominator": TESTS_25, "plan": round(G_PLAN, 2), "gap_tests": GAP_TESTS}
print(f"headline {HEADLINE['number']:+.2f}%  ({HEADLINE['numerator']:,} / {HEADLINE['denominator']:,}),  plan {HEADLINE['plan']:+.2f}%,  gap {HEADLINE['gap_tests']:,} tests")
print(f"notebook ran start to finish in {ELAPSED:.0f} seconds")
check("T4: the notebook runs top to bottom in under ten minutes", ELAPSED < 600, f"{ELAPSED:.0f}s")
check("the headline is a finite number", np.isfinite(HEADLINE["number"]))
check("the headline reproduces Dr Menon's 5 percent", abs(HEADLINE["number"] - 5.0) < 0.2)
res_checks = checks.summary()
display(res_checks[~res_checks.passed] if (~res_checks.passed).any() else pd.DataFrame({"all checks": ["passed"]}))